# Notebook 02 — Fine-tuning YOLOv8 (Detektor Kendaraan)
**Proyek:** Sistem Pemantauan & Prediksi Kemacetan Lalu Lintas Kota berbasis IoT + Big Data (Lambda Architecture)

## Posisi notebook ini
```
 Video CCTV ──► [ YOLOv8 (notebook ini) ] ──► tracking ──► fitur arus/kepadatan/kecepatan ──► Model A ──► dashboard
                  "apa & di mana kendaraannya?"     (notebook 03)
```
Speed layer membutuhkan "mata" yang bisa mengenali kendaraan di setiap frame video. YOLOv8 bawaan (dilatih di COCO) mengenal mobil, motor, bus, truk — tapi **tidak mengenal bajaj/becak** dan kurang terlatih di lalu lintas padat Asia. Karena itu kita **fine-tune**: melanjutkan pelatihan model yang sudah pintar dengan data yang mirip kondisi kita.

## Yang akan kamu kuasai
1. Memverifikasi integritas dataset (pelajaran dari upload yang rusak di notebook 01).
2. **Menggabungkan kelas** agar sesuai kebutuhan proyek dan mengurangi ketidakseimbangan.
3. Fine-tuning YOLOv8 di GPU Colab, **dengan checkpoint di Drive** supaya bisa dilanjutkan bila Colab terputus.
4. Membaca metrik deteksi: **precision, recall, mAP50, mAP50-95**.
5. Uji coba **tracking** di video — jembatan ke notebook 03.

> ⚠️ **Wajib GPU.** Menu **Runtime → Ubah jenis runtime → T4 GPU** sebelum menjalankan sel pertama.

## 0. Setup

In [ ]:
import sys, os, shutil, glob, hashlib, json, random, time, zipfile
IN_COLAB = "google.colab" in sys.modules
%pip install -q ultralytics lap     # pesan "restart the kernel" boleh diabaikan

import torch
GPU = torch.cuda.is_available()
print("Colab:", IN_COLAB, "| GPU:", torch.cuda.get_device_name(0) if GPU else "TIDAK ADA")
if IN_COLAB and not GPU:
    print("\n⚠️  Ganti runtime ke GPU dulu: Runtime → Ubah jenis runtime → T4 GPU, lalu jalankan ulang dari atas.")

### Lokasi file
Upload **satu file zip** ke Drive (jangan lewat panel File Colab):
```
MyDrive/Projek Akhir/Dataset/traffic_roboflow_yolov8.zip
```
Kenapa zip? Mengunggah 23.760 file kecil satu per satu sangat lambat dan rawan gagal. Satu file besar jauh lebih cepat dan bisa dicek keutuhannya dengan **MD5**.

Hasil training disimpan **langsung ke Drive** (`output/02_yolo/runs/`) — kalau Colab terputus di tengah jalan, progres tidak hilang.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Projek Akhir"     # <-- sesuaikan bila perlu
    WORK = "/content"
else:
    BASE = os.environ.get("PROJ_BASE", ".")
    WORK = os.environ.get("PROJ_WORK", "./work")

DATA_DIR = f"{BASE}/Dataset"
ZIP_PATH = os.environ.get("NB_ZIP", f"{DATA_DIR}/traffic_roboflow_yolov8.zip")
EXPECTED_MD5 = os.environ.get("NB_MD5", "34258e3817e6b66f93eb0e07502ab424")    # MD5 file zip di laptop
OUT_DIR  = f"{BASE}/output/02_yolo"
RUNS_DIR = f"{OUT_DIR}/runs"
DS_RAW   = f"{WORK}/datasets/roboflow"      # hasil ekstrak (disk lokal Colab, cepat)
DS       = f"{WORK}/datasets/traffic7"      # dataset dengan kelas yang sudah digabung
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(ZIP_PATH), f"Zip tidak ditemukan: {ZIP_PATH}"
print("Zip   :", ZIP_PATH, f"({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)")
print("Output:", OUT_DIR)

## 1. Ambil & verifikasi dataset
**MD5** adalah "sidik jari" file: satu byte saja berubah, MD5-nya berubah total. Kita bandingkan MD5 zip di Drive dengan MD5 yang dihitung di laptop. Kalau beda → file rusak saat upload, **berhenti dan upload ulang**. Inilah yang menyelamatkan kita di notebook 01.

In [ ]:
local_zip = f"{WORK}/traffic_roboflow_yolov8.zip"
if os.path.abspath(ZIP_PATH) != os.path.abspath(local_zip):
    shutil.copy(ZIP_PATH, local_zip)          # salin ke disk lokal: ekstrak jauh lebih cepat daripada dari Drive

h = hashlib.md5()
with open(local_zip, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
md5 = h.hexdigest()
print("MD5 file  :", md5)
print("MD5 laptop:", EXPECTED_MD5 or "(tidak diisi)")
if EXPECTED_MD5:
    assert md5 == EXPECTED_MD5, "❌ MD5 tidak cocok — file rusak saat upload. Upload ulang zip-nya."
    print("✅ File utuh")

for d in (DS_RAW, DS):
    if os.path.exists(d): shutil.rmtree(d)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(f"{WORK}/datasets")
for s in ["train", "valid", "test"]:
    print(f"{s:5s}: {len(os.listdir(f'{DS_RAW}/{s}/images')):6,} gambar | {len(os.listdir(f'{DS_RAW}/{s}/labels')):6,} label")

## 2. Memahami label YOLO
Setiap gambar punya file `.txt` dengan satu baris per objek:
```
<id_kelas> <x_tengah> <y_tengah> <lebar> <tinggi>      ← semua dinormalisasi 0–1
```
Contoh `2 0.51 0.43 0.06 0.08` = kelas ke-2, kotak di tengah gambar, lebarnya 6% gambar.

In [ ]:
import yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from collections import Counter

ORIG = yaml.safe_load(open(f"{DS_RAW}/data.yaml"))["names"]
print("Kelas asli:", ORIG)

def class_counts(label_dir, names):
    c = Counter()
    for f in os.listdir(label_dir):
        for line in open(os.path.join(label_dir, f)):
            if line.strip(): c[int(line.split()[0])] += 1
    return pd.Series({names[k]: v for k, v in sorted(c.items())})

orig_dist = class_counts(f"{DS_RAW}/train/labels", ORIG)
print("\nJumlah kotak per kelas (train):"); print((orig_dist / orig_dist.sum() * 100).round(2).astype(str) + " %")

In [ ]:
def show(img_path, label_path, names, ax):
    img = cv2.imread(img_path)[..., ::-1].copy(); H, W = img.shape[:2]
    for line in open(label_path):
        c, x, y, w, h = line.split(); c = int(c); x, y, w, h = map(float, (x, y, w, h))
        p0, p1 = (int((x - w / 2) * W), int((y - h / 2) * H)), (int((x + w / 2) * W), int((y + h / 2) * H))
        color = tuple(int(v) for v in plt.cm.tab10(c % 10, bytes=True)[:3])
        cv2.rectangle(img, p0, p1, color, 2); cv2.putText(img, names[c], (p0[0], p0[1] - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    ax.imshow(img); ax.axis("off")

random.seed(1)
samples = random.sample(os.listdir(f"{DS_RAW}/test/images"), 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for f, ax in zip(samples, axes):
    show(f"{DS_RAW}/test/images/{f}", f"{DS_RAW}/test/labels/{f[:-4]}.txt", ORIG, ax)
plt.tight_layout(); plt.show()

## 3. Menggabungkan kelas
Dua masalah dengan 9 kelas asli:
- **Terlalu timpang:** `Truck` hanya ~0,1% dan `Cycle` ~0,3% kotak → model hampir tidak belajar kelas itu.
- **Tidak semuanya relevan:** untuk menghitung arus lalu lintas, yang penting adalah *jenis beban jalan*, bukan detail tiap jenis.

Jadi kita petakan ke 7 kelas yang bermakna bagi lalu lintas Indonesia (hasil pengecekan visual sampel gambar):

| Kelas asli | Kelas baru | Padanan Indonesia |
|---|---|---|
| Bike | `motorcycle` | sepeda motor |
| Car | `car` | mobil |
| Bus | `bus` | bus |
| Truck, Mini-Truck | `truck` | truk / pikap |
| Cng | `auto_rickshaw` | bajaj |
| Rickshaw, Cycle | `nonmotorized` | becak, sepeda |
| People | `person` | pejalan kaki (tidak dihitung sebagai arus, tapi membantu model tidak salah deteksi) |

Pembagian ini juga memudahkan konversi ke **satuan mobil penumpang (smp/PCU)** di notebook 03.

**Trik hemat disk:** gambar tidak disalin — folder `images` cukup **dipindahkan** (instan, tanpa tambahan disk). Hanya file label yang ditulis ulang.

> 💡 *Cerita debugging:* versi awal notebook ini memakai *symlink* untuk folder gambar. Ternyata Ultralytics "mengikuti" symlink ke lokasi aslinya, lalu membaca **label lama** di sebelahnya → error `Label class 7 exceeds dataset class count`. Pelajaran: YOLO mencari label dengan mengganti `/images/` → `/labels/` pada path **asli** gambar.

In [ ]:
NEW = ["motorcycle", "car", "bus", "truck", "auto_rickshaw", "nonmotorized", "person"]
MAPPING = {"Bike": "motorcycle", "Car": "car", "Bus": "bus", "Truck": "truck", "Mini-Truck": "truck",
           "Cng": "auto_rickshaw", "Rickshaw": "nonmotorized", "Cycle": "nonmotorized", "People": "person"}
assert set(MAPPING) == set(ORIG), "Kelas dataset berbeda dari yang diharapkan — cek data.yaml"
ID_MAP = {ORIG.index(k): NEW.index(v) for k, v in MAPPING.items()}

for s in ["train", "valid", "test"]:
    os.makedirs(f"{DS}/{s}", exist_ok=True)
    if not os.path.exists(f"{DS}/{s}/images"):
        shutil.move(f"{DS_RAW}/{s}/images", f"{DS}/{s}/images")
    if os.path.exists(f"{DS}/{s}/labels"): shutil.rmtree(f"{DS}/{s}/labels")
    os.makedirs(f"{DS}/{s}/labels")
    for f in os.listdir(f"{DS_RAW}/{s}/labels"):
        lines = []
        for line in open(f"{DS_RAW}/{s}/labels/{f}"):
            p = line.split()
            if len(p) == 5: lines.append(" ".join([str(ID_MAP[int(p[0])])] + p[1:]))
        open(f"{DS}/{s}/labels/{f}", "w").write("\n".join(lines) + ("\n" if lines else ""))

yaml.safe_dump({"path": DS, "train": "train/images", "val": "valid/images", "test": "test/images",
                "nc": len(NEW), "names": NEW}, open(f"{DS}/data.yaml", "w"), sort_keys=False)
print(open(f"{DS}/data.yaml").read())

new_dist = class_counts(f"{DS}/train/labels", NEW)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
(orig_dist / orig_dist.sum() * 100).plot.barh(ax=axes[0], color="#9E9E9E", title="Sebelum (9 kelas)")
(new_dist / new_dist.sum() * 100).plot.barh(ax=axes[1], color="#4C78A8", title="Sesudah (7 kelas)")
for ax in axes: ax.set_xlabel("% kotak")
plt.tight_layout(); plt.show()

## 4. Training
### Konsep singkat
- **Transfer learning:** mulai dari `yolov8s.pt` yang sudah dilatih jutaan gambar COCO → model sudah paham "bentuk kendaraan", kita tinggal mengajari kelas dan sudut pandang baru. Jauh lebih cepat & akurat daripada dari nol.
- **Epoch:** satu putaran penuh melihat seluruh data train.
- **Patience (early stopping):** kalau performa di data *valid* tidak membaik selama N epoch, training berhenti sendiri → mencegah *overfitting* dan menghemat kuota GPU.
- **`imgsz=640`:** objek di dataset ini kecil (median ~35 piksel), jadi resolusi jangan diturunkan.

### Mode
- `QUICK_TEST = True` → 3 epoch dengan 10% data (±10 menit). **Jalankan ini dulu** untuk memastikan semua lancar.
- `QUICK_TEST = False` → training penuh 25 epoch (perkiraan **2–3 jam di T4**). Checkpoint tersimpan di Drive setiap epoch.

**Kalau Colab terputus saat training penuh:** sambungkan lagi, jalankan semua sel dari atas sampai sel ini. Kode di bawah otomatis **melanjutkan** dari checkpoint terakhir, bukan mulai dari awal.

In [ ]:
QUICK_TEST = True        # <-- ubah ke False untuk training penuh

TEST_MODE = os.environ.get("NB_TEST") == "1"            # hanya untuk uji notebook tanpa GPU
MODEL    = "yolov8n.pt" if TEST_MODE else "yolov8s.pt"
IMGSZ    = 320 if TEST_MODE else 640
EPOCHS   = 1 if TEST_MODE else (3 if QUICK_TEST else 25)
FRACTION = 1.0 if TEST_MODE else (0.1 if QUICK_TEST else 1.0)
RUN_NAME = "quick_test" if (QUICK_TEST or TEST_MODE) else "yolov8s_traffic7"
print(f"Model {MODEL} | imgsz {IMGSZ} | epochs {EPOCHS} | data {FRACTION:.0%} | run '{RUN_NAME}'")

In [ ]:
from ultralytics import YOLO

run_dir = f"{RUNS_DIR}/{RUN_NAME}"
last_ckpt = f"{run_dir}/weights/last.pt"
t0 = time.time()

if os.path.exists(last_ckpt) and RUN_NAME != "quick_test":
    print("▶ Checkpoint ditemukan — melanjutkan training:", last_ckpt)
    try:
        YOLO(last_ckpt).train(resume=True)
    except AssertionError as e:
        print("Training sebelumnya sudah selesai:", e)
else:
    model = YOLO(MODEL)
    model.train(
        data=f"{DS}/data.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=16, fraction=FRACTION,
        patience=8, cos_lr=True, close_mosaic=5, seed=42,
        workers=2, device=0 if GPU else "cpu",
        project=RUNS_DIR, name=RUN_NAME, exist_ok=True, plots=True, verbose=False)

print(f"\nSelesai dalam {(time.time() - t0) / 60:.1f} menit")

### Membaca kurva training
`results.png` menampilkan loss dan metrik per epoch:
- **Loss train & val turun bersama** → model belajar dengan sehat.
- **Loss train turun tapi val naik** → *overfitting* (model menghafal, bukan memahami).
- **mAP50-95 naik lalu mendatar** → model sudah jenuh; menambah epoch tidak banyak membantu.

In [ ]:
res_img = f"{run_dir}/results.png"
if os.path.exists(res_img):
    plt.figure(figsize=(14, 7)); plt.imshow(plt.imread(res_img)); plt.axis("off"); plt.show()
pd.read_csv(f"{run_dir}/results.csv").rename(columns=str.strip).tail(5)

## 5. Evaluasi di data test
Metrik deteksi yang perlu kamu pahami (dan tulis di laporan):
- **Precision** — dari semua kotak yang ditebak model, berapa persen yang benar? (rendah = banyak deteksi palsu)
- **Recall** — dari semua kendaraan yang ada, berapa persen yang ketemu? (rendah = banyak kendaraan terlewat → hitungan arus terlalu kecil)
- **mAP50** — rata-rata presisi bila kotak dianggap benar saat tumpang tindih ≥ 50% dengan kotak asli.
- **mAP50-95** — versi ketat (rata-rata ambang 50%–95%). Ini metrik utama di paper.

Untuk proyek ini, **recall kendaraan** paling penting, karena kendaraan yang terlewat = arus dan kepadatan yang dilaporkan terlalu rendah.

> ⚠️ **Jujur di laporan:** split dataset Roboflow ini berbasis frame acak dari video yang sama, sehingga frame bertetangga bisa muncul di train dan test. Angka test kemungkinan **lebih optimistis** dari performa di video baru. Uji sesungguhnya ada di bagian 6 dan notebook 03.

In [ ]:
best_ckpt = f"{run_dir}/weights/best.pt"
best = YOLO(best_ckpt)
m = best.val(data=f"{DS}/data.yaml", split="test", imgsz=IMGSZ, batch=16, device=0 if GPU else "cpu",
             project=RUNS_DIR, name=f"{RUN_NAME}_test", exist_ok=True, plots=True, verbose=False)

per_class = pd.DataFrame({
    "kelas":     [NEW[i] for i in m.box.ap_class_index],
    "precision": m.box.p, "recall": m.box.r,
    "mAP50":     m.box.ap50, "mAP50-95": m.box.ap,
}).round(3)
summary = {"precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map}
print("RATA-RATA SEMUA KELAS:", {k: round(float(v), 3) for k, v in summary.items()})
per_class

In [ ]:
cm_img = os.path.join(m.save_dir, "confusion_matrix_normalized.png")
if os.path.exists(cm_img):
    plt.figure(figsize=(8, 7)); plt.imshow(plt.imread(cm_img)); plt.axis("off"); plt.show()

**Cara membaca confusion matrix:** diagonal = tebakan benar. Nilai di luar diagonal menunjukkan kelas yang tertukar (mis. `truck` ↔ `bus` wajar tertukar karena labelnya pun kadang keliru). Kolom/baris **background** = objek terlewat atau deteksi palsu.

In [ ]:
random.seed(7)
imgs = random.sample(sorted(glob.glob(f"{DS}/test/images/*.jpg")), 4)
preds = best.predict(imgs, imgsz=IMGSZ, conf=0.3, verbose=False, device=0 if GPU else "cpu")
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for r, ax in zip(preds, axes.ravel()):
    ax.imshow(r.plot(line_width=1, font_size=8)[..., ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Uji tracking di video (jembatan ke notebook 03)
Deteksi saja belum cukup — untuk menghitung arus dan kecepatan kita harus tahu bahwa mobil di frame 10 dan frame 11 adalah **mobil yang sama**. Itu tugas **tracker** (ByteTrack): setiap kendaraan mendapat **ID unik** yang konsisten antar frame.

Upload video uji (kamera diam) ke `MyDrive/Projek Akhir/Dataset/video_uji.mp4`. Sel ini memproses 20 detik pertama, menghitung kendaraan unik per kelas, dan menyimpan video beranotasi.

In [ ]:
VIDEO = os.environ.get("NB_VIDEO", f"{DATA_DIR}/video_uji.mp4")
MAX_SECONDS = 20

if not os.path.exists(VIDEO):
    print("Video uji belum ada di", VIDEO, "— lewati sel ini atau upload dulu.")
else:
    cap = cv2.VideoCapture(VIDEO); fps = cap.get(cv2.CAP_PROP_FPS) or 25; cap.release()
    max_frames = int(fps * MAX_SECONDS)
    out_path = f"{OUT_DIR}/demo_tracking.mp4"
    writer, seen = None, {}
    t0 = time.time()
    for i, r in enumerate(best.track(source=VIDEO, tracker="bytetrack.yaml", stream=True, imgsz=IMGSZ,
                                     conf=0.3, persist=True, verbose=False, device=0 if GPU else "cpu")):
        if r.boxes.id is not None:
            for tid, cls in zip(r.boxes.id.int().tolist(), r.boxes.cls.int().tolist()):
                if NEW[cls] != "person":                 # pejalan kaki bukan bagian arus kendaraan
                    seen.setdefault(tid, NEW[cls])
        frame = r.plot(line_width=1, font_size=8)
        scale = 1280 / max(frame.shape[:2])
        if scale < 1: frame = cv2.resize(frame, None, fx=scale, fy=scale)
        if writer is None:
            writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame.shape[1], frame.shape[0]))
        writer.write(frame)
        if i + 1 >= max_frames: break
    writer.release()
    print(f"{i + 1} frame diproses dalam {time.time() - t0:.0f} detik ({(i + 1) / (time.time() - t0):.1f} FPS)")
    print("Kendaraan unik per kelas:", dict(Counter(seen.values())))
    print("Video beranotasi:", out_path)

**Perhatikan di video beranotasi:** apakah ID kendaraan tetap sama saat bergerak? ID yang sering berganti (*ID switch*) akan membuat hitungan arus berlebih — ini yang akan kita tangani di notebook 03 dengan **garis hitung virtual**.

## 7. Simpan model
File `best.pt` adalah "otak" speed layer — dipakai di notebook 03 dan di backend dashboard.

In [ ]:
final_name = "yolov8s_traffic7.pt" if RUN_NAME == "yolov8s_traffic7" else f"{RUN_NAME}.pt"
shutil.copy(best_ckpt, f"{OUT_DIR}/{final_name}")
json.dump({"model": final_name, "base": MODEL, "imgsz": IMGSZ, "epochs_planned": EPOCHS, "classes": NEW,
           "class_mapping": MAPPING, "test_metrics": {k: round(float(v), 4) for k, v in summary.items()},
           "per_class": per_class.to_dict(orient="records")},
          open(f"{OUT_DIR}/{final_name.replace('.pt', '_info.json')}", "w"), indent=2)
print("Tersimpan di", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)

## 8. Ringkasan
| Hasil | Lokasi |
|---|---|
| Model terlatih | `output/02_yolo/yolov8s_traffic7.pt` |
| Metrik + mapping kelas (untuk laporan) | `output/02_yolo/yolov8s_traffic7_info.json` |
| Kurva training, confusion matrix, contoh prediksi | `output/02_yolo/runs/` |
| Video demo tracking | `output/02_yolo/demo_tracking.mp4` |

**Checklist sebelum lanjut:**
- [ ] Sudah menjalankan `QUICK_TEST = True` tanpa error
- [ ] Sudah menjalankan training penuh (`QUICK_TEST = False`)
- [ ] File `yolov8s_traffic7.pt` ada di Drive

**Catatan lisensi untuk laporan:** dataset Roboflow *Traffic Congestion Detection* (tishas-workspace) — CC BY 4.0. Ultralytics YOLOv8 — AGPL-3.0 (aman untuk tugas akademik).

---
**Selanjutnya → Notebook 03:** tracking + garis hitung virtual di video UCSD → fitur arus, kepadatan, kecepatan → **Model A** (klasifikasi level kemacetan).